# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Data Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² dataset via the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You'll discover data access, understand the schema structure using entity `@id`s, and perform exploratory analysis.

### Dataset Source

**FAIR² Croissant schema URL:**  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and discover recordsets and fields using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata  # `metadata` is a DatasetMetadata object
print("Dataset loaded successfully!\n")
print(f"Dataset name: {metadata_obj.name}\n")
print(f"Description: {metadata_obj.description}\n")
print(f"Version: {metadata_obj.version}")

## 2. Data Overview

Identify available record sets, their `@id`s, and the fields they contain. All Croissant entities are referenced by their `@id` for clarity and reproducibility.

_Let's enumerate the record sets and their fields._

In [ ]:
# List available record sets and their fields by @id

if not metadata_obj.record_sets():
    print("No record sets are defined in the metadata; attempting to auto-discover via dataset.records().")
else:
    print("Record sets in this dataset:")
    for rs in metadata_obj.record_sets():
        print(f"- Record set: {rs.name} (id: {rs.id})")
        if rs.fields:
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.name} (id: {f.id})")
        else:
            print("  No explicit fields in this record set.")

# For exploratory purposes, try to list the record set @ids discovered by .records()
print("\nAttempting to discover available record sets via dataset.records():")
try:
    known_record_sets = set()
    # Try a few: dataset.records() yields dicts with a '_record_set' key
    for i, rec in enumerate(dataset.records()):
        # rec['_record_set'] is the record set id
        rs_id = rec.get('_record_set', None)
        if rs_id:
            known_record_sets.add(rs_id)
        if i > 100:
            break
    if known_record_sets:
        print("Record set @ids found:")
        for rs_id in sorted(known_record_sets):
            print(f"  {rs_id}")
    else:
        print("No record set @id found via dataset.records().")
except Exception as e:
    print(f"Encountered an error while exploring records: {e}")

## 3. Data Extraction

Load data from the available record sets using their `@id`s into Pandas DataFrames. Use the record set and field `@id`s found above.

**Note:** For this dataset, you may need to inspect the available record sets (previous output) and update the code with the appropriate `@id`s.

In [ ]:
# Prepare to extract all available record sets.

# Option 1: Get record set @ids from metadata
record_set_ids = [rs.id for rs in metadata_obj.record_sets()] if metadata_obj.record_sets() else []

# Option 2: If not found, use the discovered record set ids
if not record_set_ids:
    # Try to access from earlier cell's known_record_sets (set), else hardcode empty list
    try:
        record_set_ids = list(known_record_sets)
    except NameError:
        record_set_ids = []

print(f"Record sets to extract: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (fields):")
        print(df.columns.tolist())
        display(df.head(2))
    else:
        print("No records found for this record set.")

# If no record sets found, try to extract all records (dataset.records() with no filter)
if not dataframes:
    print("\nNo organized record sets found. Attempting flat extraction of all available records.")
    all_records = list(dataset.records())
    if all_records:
        df = pd.DataFrame(all_records)
        dataframes['all'] = df
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        display(df.head(2))
    else:
        print("No records found in dataset.")

## 4. Exploratory Data Analysis (EDA)

Now, let's analyze and transform a numeric field. We'll demonstrate how to filter on a numeric column using its field `@id`, normalize it, and group by another categorical field using its `@id` as well.

In [ ]:
# Select a DataFrame and fields for analysis.

# Try to pick the first available DataFrame for demonstration
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"\nUsing DataFrame for record set: {df_key}")
    numeric_field_id = None
    group_field_id = None
    # Try to find a plausible numeric field (e.g., 'log_likelihood', 'coeff', etc.)
    for col in df.columns:
        if "log" in col.lower() or "coefficient" in col.lower() or "std" in col.lower() or "value" in col.lower():
            numeric_field_id = col
            break
    # Try for a plausible categorical field ('ward', 'gender', etc.)
    for col in df.columns:
        if "ward" in col.lower() or "gender" in col.lower() or "county" in col.lower():
            group_field_id = col
            break
    if not numeric_field_id:
        # Default to first float/int-like field
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not group_field_id and df.columns.size > 1:
        group_field_id = df.columns[1]
    print(f"Numeric field (for filtering/normalization): {numeric_field_id}")
    print(f"Group field (optional): {group_field_id}")

    # Set an example threshold
    threshold = None
    if numeric_field_id is not None:
        try:
            if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
                threshold = df[numeric_field_id].mean()
            else:
                # Try converting to numeric
                df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
                threshold = df[numeric_field_id].mean()
        except Exception:
            threshold = 0
    else:
        print("No numeric field found—cannot proceed with EDA.")

    # Filter, normalize, and group data
    if numeric_field_id and threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (field @id: {numeric_field_id}):")
        display(filtered_df.head(2))

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(2))

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head(2))
    else:
        print("Could not perform EDA: missing suitable numeric field.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field by group using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if df and numeric field are available
if 'filtered_df' in locals() and filtered_df.shape[0] > 0 and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    if group_field_id and group_field_id in filtered_df.columns:
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
    else:
        sns.histplot(filtered_df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Visualization not possible: missing filtered DataFrame or numeric field.")

## 6. Conclusion

- We explored the FAIR² dataset, using Croissant's metadata to identify record sets and their fields by `@id`.
- Data extraction used selected record sets, enabling programmatic, reproducible analysis.
- Exploratory Data Analysis demonstrated filtering, normalization, grouping, and visualization, all referencing Croissant schema `@id` fields.
- For further analysis, refer to the Croissant schema or dataset documentation for specific field meanings, and see the [mlcroissant docs](https://github.com/mlcommons/croissant) for more advanced workflows.